# Business Data Quality POC – Retail Use Case

> **Vision**: An AI agent that acts as a data scientist – running correlation analysis, EDA,
> and statistical tests grounded in *business knowledge*, not just technical outlier detection.

---

## 10 Business Data Quality Scenarios

| # | Scenario | Key Insight |
|---|----------|-------------|
| 1 | **Contextual Sales Anomaly** | High sales backed by high orders is NOT an anomaly |
| 2 | **Returns-Rate Spike** | Returns spike on a normal-sales day IS anomalous |
| 3 | **Inventory Reconciliation** | Closing stock must equal opening − sold + restocked |
| 4 | **Ghost / Phantom Inventory** | Closing stock impossibly higher than opening + restock |
| 5 | **Discount-Volume Relationship** | Deep discount with no sales uplift is a business anomaly |
| 6 | **Cross-Store Price Inconsistency** | Same product should not be 85 % more expensive in one store |
| 7 | **Selling Below Cost (Margin Erosion)** | Negative unit margin violates a hard business rule |
| 8 | **CLV / AOV Inconsistency** | Premium customers can't have CLV 100× their average order |
| 9 | **Seasonal Pattern Violation** | Sales should follow known seasonal indices |
| 10 | **New-Premium Churn Paradox** | Newly acquired Premium customers with >90% churn risk are suspect |

## 0 – Environment Setup

In [ ]:
# Install dependencies (run once)
# !pip install langgraph langchain langchain-openai pandas numpy scipy matplotlib seaborn

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Add repo root to path so we can import the backend modules
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Optional: set your OpenAI key here to enable LLM-powered narrative reports
# os.environ['OPENAI_API_KEY'] = 'sk-...'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

---
## 1 – Generate Sample Retail Dataset

The generator creates **6 interrelated tables** with **11 embedded data quality issues**.

In [ ]:
from data.generate_retail_dataset import generate_all

DATA_DIR = os.path.join(REPO_ROOT, 'data')
datasets = generate_all(output_dir=DATA_DIR)

products_df    = datasets['products']
orders_df      = datasets['orders']
sales_df       = datasets['sales']
inventory_df   = datasets['inventory']
customers_df   = datasets['customers']
supply_df      = datasets['supply_chain']

In [ ]:
# Quick overview
for name, df in datasets.items():
    print(f'{name:15s}: {len(df):>8,} rows  ×  {len(df.columns)} cols')

In [ ]:
print('=== Sales sample ===')
display(sales_df.head(3))
print('=== Orders sample ===')
display(orders_df.head(3))
print('=== Inventory sample ===')
display(inventory_df.head(3))

---
## 2 – Exploratory Data Analysis

In [ ]:
# Monthly revenue trend
sales_df['date'] = pd.to_datetime(sales_df['date'])
monthly = sales_df.groupby(sales_df['date'].dt.to_period('M'))['sales_amount'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

monthly.plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Monthly Revenue')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=45)

# Return rate by category
sales_df = sales_df.merge(products_df[['product_id','category']], on='product_id', how='left')
return_rate = (
    sales_df.groupby('category')
    .apply(lambda x: x['returns_qty'].sum() / max(x['sales_qty'].sum(), 1))
    .sort_values(ascending=False)
)
return_rate.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Return Rate by Category')
axes[1].set_ylabel('Return Rate')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of discounts and unit prices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sales_df['discount_pct'].hist(bins=40, ax=axes[0], color='mediumpurple', edgecolor='white')
axes[0].set_title('Discount % Distribution')

for pid, grp in sales_df.groupby('product_id'):
    grp['unit_price'].hist(bins=30, ax=axes[1], alpha=0.4, label=pid)
axes[1].set_title('Unit Price Distribution by Product')
axes[1].legend(fontsize=7)

sales_df['gross_margin'].hist(bins=50, ax=axes[2], color='darkcyan', edgecolor='white')
axes[2].axvline(0, color='red', linewidth=1.5, linestyle='--', label='Break-even')
axes[2].set_title('Gross Margin Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 3 – Run the Business Data Quality Agent (LangGraph)

The agent orchestrates four tool nodes:
1. **Anomaly Detector** – contextual anomalies
2. **Correlation Analyser** – cross-metric business relationships
3. **Statistical Tester** – hypothesis tests (t-test, KS, chi-square)
4. **Business Rule Validator** – hard rule violations

In [ ]:
from backend.agents import run_dq_analysis

result = run_dq_analysis(
    sales_df=sales_df,
    orders_df=orders_df,
    inventory_df=inventory_df,
    customers_df=customers_df,
    supply_df=supply_df,
)

print(result['final_report'])

In [ ]:
# LLM narrative (requires OPENAI_API_KEY)
print(result.get('llm_narrative', 'LLM narrative not generated.'))

---
## 4 – Scenario Deep Dives

### Scenario 1 – Contextual Sales Anomaly (Sales vs Orders)

> **Business Logic**: A single-column outlier detector would flag any day with unusually high
> sales. But if orders were *also* unusually high that day, the sales spike is fully explained
> — it is NOT a data quality issue. Our agent only raises an alert when sales spike WITHOUT
> a corresponding order spike.

In [ ]:
from backend.agents.tools.anomaly_detection import sales_vs_orders_anomaly

anomaly_df = sales_vs_orders_anomaly(sales_df, orders_df)

true_anom   = anomaly_df[anomaly_df['anomaly_type'] == 'SALES_WITHOUT_ORDER_BACKING']
explained   = anomaly_df[anomaly_df['anomaly_type'] == 'EXPLAINED_SPIKE_HIGH_ORDERS']

print(f'True anomalies  (sales spike with NO order backing): {len(true_anom)}')
print(f'Explained spikes (sales spike backed by high orders): {len(explained)}')

# Visualise for P001 / S002
agg_s = sales_df.groupby(['date','store_id','product_id'])['sales_qty'].sum().reset_index()
agg_o = orders_df.groupby(['date','store_id','product_id'])['orders_received'].sum().reset_index()
merged = agg_s.merge(agg_o, on=['date','store_id','product_id'])

focus = merged[(merged['store_id']=='S002') & (merged['product_id']=='P001')].copy()
focus['date'] = pd.to_datetime(focus['date'])
focus = focus.sort_values('date')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(focus['date'], focus['orders_received'], label='Orders Received', color='steelblue', alpha=0.7)
ax.plot(focus['date'], focus['sales_qty'],       label='Sales Qty',       color='darkorange')

# Highlight true anomaly window
ax.axvspan(pd.Timestamp('2023-06-01'), pd.Timestamp('2023-06-10'),
           alpha=0.15, color='red', label='Injected Anomaly Window')

ax.set_title('Store S002 | Product P001 – Sales vs Orders (June anomaly injected)')
ax.set_xlabel('Date')
ax.set_ylabel('Units')
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop flagged rows (true anomalies):')
display(true_anom[['date','store_id','product_id','sales_qty','orders_received','anomaly_type']].head(10))

### Scenario 2 – Returns-Rate Spike Not Driven by Sales

In [ ]:
from backend.agents.tools.anomaly_detection import returns_anomaly

ret_df = returns_anomaly(sales_df)
print(f'Return-rate anomaly rows detected: {len(ret_df)}')

focus_r = sales_df[
    (sales_df['store_id']=='S003') & (sales_df['product_id']=='P002')
].copy().sort_values('date')
focus_r['date'] = pd.to_datetime(focus_r['date'])
focus_r['return_rate'] = focus_r['returns_qty'] / focus_r['sales_qty'].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(focus_r['date'], focus_r['return_rate'], color='tomato', label='Return Rate')
ax.axhline(focus_r['return_rate'].mean(), color='gray', linestyle='--', label='Mean')
ax.axvspan(pd.Timestamp('2023-09-01'), pd.Timestamp('2023-09-07'),
           alpha=0.15, color='red', label='Injected Returns Spike')
ax.set_title('Store S003 | Product P002 – Daily Return Rate')
ax.set_xlabel('Date')
ax.set_ylabel('Return Rate')
ax.legend()
plt.tight_layout()
plt.show()

### Scenario 3 & 4 – Inventory Reconciliation & Ghost Inventory

In [ ]:
from backend.agents.tools.anomaly_detection import inventory_accuracy_anomaly

inv_issues = inventory_accuracy_anomaly(inventory_df)
print(inv_issues['anomaly_type'].value_counts())
display(inv_issues[inv_issues['anomaly_type']=='INVENTORY_RECONCILIATION_ERROR']
        [['date','store_id','product_id','opening_stock','units_sold',
          'restocked_qty','closing_stock','expected_closing','reconciliation_diff']].head(10))

### Scenario 5 – Discount-Volume Relationship Anomaly

In [ ]:
from backend.agents.tools.correlation_analysis import discount_revenue_correlation

disc_result = discount_revenue_correlation(sales_df)
print(disc_result['summary'])
display(disc_result['flagged_pairs'][['store_id','product_id','pearson_r','avg_discount','issue']])

In [ ]:
# Visualise for P006 (Smartphone) where a 45% discount was injected
p6 = sales_df[sales_df['product_id']=='P006'].copy()
p6['date'] = pd.to_datetime(p6['date'])
p6 = p6.sort_values('date')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for sid, grp in p6.groupby('store_id'):
    axes[0].plot(grp['date'], grp['discount_pct'], alpha=0.6, label=sid)
axes[0].set_title('P006 Smartphone – Discount % over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Discount %')
axes[0].legend(fontsize=8)

axes[1].scatter(p6['discount_pct'], p6['sales_qty'], alpha=0.3, color='mediumpurple')
axes[1].set_title('P006 – Discount % vs Sales Qty')
axes[1].set_xlabel('Discount %')
axes[1].set_ylabel('Sales Qty')

plt.tight_layout()
plt.show()

### Scenario 6 – Cross-Store Price Inconsistency

In [ ]:
from backend.agents.tools.correlation_analysis import price_consistency_across_stores

price_result = price_consistency_across_stores(sales_df)
print(price_result['summary'])

pivot = (
    price_result['price_table']
    .pivot(index='product_id', columns='store_id', values='avg_price')
)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Average Unit Price per Product × Store (anomalous prices appear bright red)')
plt.tight_layout()
plt.show()

display(price_result['flagged_stores'][['product_id','store_id','avg_price',
                                        'product_avg_price','z_price','issue']])

### Scenario 7 – Selling Below Cost (Margin Erosion)

In [ ]:
from backend.agents.tools.correlation_analysis import sales_margin_correlation

margin_result = sales_margin_correlation(sales_df)
print(margin_result['summary'])

neg = margin_result['negative_margin_rows']
if not neg.empty:
    display(neg[['date','store_id','product_id','unit_price','cost_per_unit',
                 'discount_pct','unit_margin','issue']].head(10))

### Scenario 8 – CLV / AOV Inconsistency

In [ ]:
from backend.agents.tools.correlation_analysis import clv_aov_consistency

clv_result = clv_aov_consistency(customers_df)
print(clv_result['summary'])
display(clv_result['segment_correlations'])

fig, ax = plt.subplots(figsize=(8, 5))
for seg, grp in customers_df.groupby('segment'):
    ax.scatter(grp['avg_order_value'], grp['clv_estimate'],
               alpha=0.5, label=seg, s=20)
# Highlight flagged
flagged_c = clv_result['flagged_customers']
if not flagged_c.empty:
    ax.scatter(flagged_c['avg_order_value'], flagged_c['clv_estimate'],
               color='red', s=60, zorder=5, marker='x', label='Flagged')
ax.set_xlabel('Avg Order Value ($)')
ax.set_ylabel('CLV Estimate ($)')
ax.set_title('Customer CLV vs Average Order Value')
ax.legend()
plt.tight_layout()
plt.show()

### Scenario 9 – Seasonal Pattern Violation

In [ ]:
from backend.agents.tools.statistical_tests import seasonal_pattern_test

seasonal_result = seasonal_pattern_test(sales_df)
print(seasonal_result['summary'])

# Plot expected vs actual for one product
sample_pid = 'P001'
sample_sid = 'S001'
subset = seasonal_result['seasonal_test_table'][
    (seasonal_result['seasonal_test_table']['product_id'] == sample_pid) &
    (seasonal_result['seasonal_test_table']['store_id']   == sample_sid)
].sort_values('month')

if not subset.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = subset['month']
    ax.plot(x, subset['expected_index'], 'b--o', label='Expected Seasonal Index')
    ax.plot(x, subset['actual_index'],   'r-o',  label='Actual Index')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                        'Jul','Aug','Sep','Oct','Nov','Dec'])
    ax.set_title(f'{sample_pid} | {sample_sid} – Seasonal Index: Actual vs Expected')
    ax.set_ylabel('Relative Sales Index')
    ax.legend()
    plt.tight_layout()
    plt.show()

### Scenario 10 – New-Premium Churn Paradox

In [ ]:
from backend.agents.tools.statistical_tests import churn_risk_paradox_test

churn_result = churn_risk_paradox_test(customers_df)
print(churn_result['summary'])
display(churn_result['contingency_table'])

fig, ax = plt.subplots(figsize=(8, 4))
premium = customers_df[customers_df['segment']=='Premium'].copy()
premium['is_new'] = premium['acquisition_month'] >= 10
colors = premium['is_new'].map({True: 'tomato', False: 'steelblue'})
ax.scatter(premium['acquisition_month'], premium['churn_risk_score'],
           c=colors, alpha=0.5, s=30)
ax.axhline(0.85, color='red', linestyle='--', label='High churn threshold (0.85)')
ax.set_xlabel('Acquisition Month')
ax.set_ylabel('Churn Risk Score')
ax.set_title('Premium Customers – Churn Risk by Acquisition Month\n(red = new, blue = established)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5 – Business Rule Violations Dashboard

In [ ]:
from backend.agents.tools.business_rules import validate_business_rules

rules_result = validate_business_rules(sales_df, orders_df, inventory_df)
print(rules_result['overall_summary'])

summary_df = rules_result['summary_by_rule']
display(summary_df)

# Bar chart of violation counts
if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(summary_df['rule_id'], summary_df['violation_count'],
                  color=['tomato','steelblue','mediumpurple','darkcyan','darkorange','seagreen'][:len(summary_df)])
    ax.set_title('Business Rule Violation Counts')
    ax.set_xlabel('Rule ID')
    ax.set_ylabel('Number of Violations')
    for bar, count in zip(bars, summary_df['violation_count']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontsize=11)
    plt.tight_layout()
    plt.show()

---
## 6 – Consolidated Issues Summary

In [ ]:
anomaly_r = result.get('anomaly_results', {}).get('summary', {})
corr_r    = result.get('correlation_results', {}).get('overall_summary', {})
stat_r    = result.get('statistical_results', {}).get('overall_summary', {})

summary_data = {
    'Category': [
        'Contextual Anomaly – True Sales Anomalies',
        'Contextual Anomaly – Explained Spikes (NOT issues)',
        'Contextual Anomaly – Return Rate Spikes',
        'Contextual Anomaly – Inventory Issues',
        'Contextual Anomaly – Fulfillment Anomalies',
        'Correlation – Discount with No Volume Gain',
        'Correlation – Price Inconsistency',
        'Correlation – Rows Sold Below Cost',
        'Correlation – CLV/AOV Mismatches',
        'Statistical – Seasonal Pattern Violations',
        'Statistical – New-Premium High Churn',
        'Statistical – Fulfillment Lead-time Anomaly',
        'Business Rules – Total Violations',
    ],
    'Count': [
        anomaly_r.get('sales_true_anomaly_count', 0),
        anomaly_r.get('explained_spike_count', 0),
        anomaly_r.get('return_anomaly_count', 0),
        anomaly_r.get('inventory_anomaly_count', 0),
        anomaly_r.get('fulfillment_anomaly_count', 0),
        corr_r.get('discount_issues', 0),
        corr_r.get('price_issues', 0),
        corr_r.get('margin_issues', 0),
        corr_r.get('clv_aov_issues', 0),
        stat_r.get('seasonal_violations', 0),
        stat_r.get('churn_paradox_count', 0),
        int(stat_r.get('fulfillment_anomaly', False)),
        rules_result.get('total_violations', 0),
    ],
}

summary_table = pd.DataFrame(summary_data)
display(summary_table)

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 7))
colors_list = ['tomato' if 'NOT' not in c else 'seagreen' for c in summary_table['Category']]
ax.barh(summary_table['Category'], summary_table['Count'], color=colors_list, edgecolor='white')
ax.set_xlabel('Issue Count')
ax.set_title('Business Data Quality Issues Detected\n(green = not a real issue, red = genuine DQ problem)')
plt.tight_layout()
plt.show()

---
## 7 – LangGraph Workflow Visualisation

In [ ]:
from backend.agents import build_dq_graph

graph = build_dq_graph()
try:
    # Requires graphviz: pip install grandalf
    graph.get_graph().print_ascii()
except Exception:
    print('Graph topology:')
    print('  START → router → [anomaly, correlation, statistical, rules] → report → END')

---
## 8 – Next Steps & Roadmap

| Priority | Enhancement | Description |
|----------|-------------|-------------|
| P1 | LLM-powered router | Replace heuristic router with GPT-4o that reads a user question and selects tools |
| P1 | Natural language findings | Each tool node posts findings as structured LangChain messages |
| P2 | Additional scenarios | Demand forecast accuracy, supplier performance drift, channel mix anomalies |
| P2 | Streaming report | Stream findings to UI as each tool node completes |
| P3 | Human-in-the-loop | Add a review node where a business analyst can confirm or reject flagged issues |
| P3 | Remediation suggestions | For each issue type, suggest automated data correction steps |
| P3 | FastAPI integration | Expose `run_dq_analysis` as a REST endpoint connected to the existing `main.py` |

---
*End of POC notebook — Business Data Quality Agentic Framework v0.1*